In [0]:
# ============================================================
# STEP 1 — FRESH FILE MANIFEST TABLE
# ============================================================

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DateType,
    TimestampType,
    IntegerType,
    LongType,
    DoubleType
)
from delta.tables import DeltaTable

In [0]:
# Existing project path — do not change to Landing or Bronze.
manifest_path = (
    "abfss://customer-sales-datalake@"
    "nestlesalesdatalake.dfs.core.windows.net/"
    "metadata/file_manifest/"
)
dbutils.widgets.text("dataset_name", "campaigns")
dbutils.widgets.text("source_name", "google_ads")
dataset_name=dbutils.widgets.get("dataset_name").lower()
source_name=dbutils.widgets.get("source_name").lower()

In [0]:
# ------------------------------------------------------------
# 3. Define the complete 32-column schema
# ------------------------------------------------------------

from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, DateType, LongType, DoubleType
manifest_schema = StructType([

    # 1–11: Core identity and existing manifest fields
    StructField("dataset", StringType(), True),
    StructField("file_name", StringType(), True),
    StructField("file_path", StringType(), True),
    StructField("batch_id", StringType(), True),
    StructField("status", StringType(), True),
    StructField("first_seen", TimestampType(), True),
    StructField("processed_at", TimestampType(), True),
    StructField("run_id", StringType(), True),
    StructField("retry_count", IntegerType(), True),
    StructField("error_message", StringType(), True),
    StructField("data_date", DateType(), True),

    # 12–16: Discovery and source metadata
    StructField("file_size_bytes", LongType(), True),
    StructField("source_modified_at", TimestampType(), True),
    StructField("content_hash", StringType(), True),
    StructField("last_seen_at", TimestampType(), True),
    StructField("discovery_run_id", StringType(), True),

    # 17–22: Claiming, attempts, and recovery
    StructField("claimed_at", TimestampType(), True),
    StructField("lease_expires_at", TimestampType(), True),
    StructField("last_attempt_at", TimestampType(), True),
    StructField("attempt_count", IntegerType(), True),
    StructField("last_success_at", TimestampType(), True),
    StructField("status_updated_at", TimestampType(), True),

    # 23–27: Data-quality and processing verification
    StructField("source_record_count", LongType(), True),
    StructField("written_record_count", LongType(), True),
    StructField("rejected_record_count", LongType(), True),
    StructField("processed_content_hash", StringType(), True),
    StructField("schema_fingerprint", StringType(), True),

    # 28–30: Destination and performance metadata
    StructField("target_path", StringType(), True),
    StructField("target_delta_version", LongType(), True),
    StructField("processing_duration_seconds", DoubleType(), True),

    # 31–32: Error tracking
    StructField("error_type", StringType(), True),
    StructField("last_error_at", TimestampType(), True)
])


In [0]:

# ------------------------------------------------------------
# 5. Verify the existing File Manifest
# ------------------------------------------------------------

manifest_df = (
    spark.read
    .format("delta")
    .load(manifest_path)
)

actual_columns = manifest_df.columns
actual_count = len(actual_columns)
record_count = manifest_df.count()

if actual_count != 32:
    raise RuntimeError(
        f"Verification failed: expected 32 columns, "
        f"found {actual_count}."
    )

print("File Manifest verified successfully.")
print("Delta path:", manifest_path)
print("Column count:", actual_count)
print("Existing record count:", record_count)

manifest_df.printSchema()
display(manifest_df)


File Manifest verified successfully.
Delta path: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/metadata/file_manifest/
Column count: 32
Existing record count: 26
root
 |-- dataset: string (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_path: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- first_seen: timestamp (nullable = true)
 |-- processed_at: timestamp (nullable = true)
 |-- run_id: string (nullable = true)
 |-- retry_count: integer (nullable = true)
 |-- error_message: string (nullable = true)
 |-- data_date: date (nullable = true)
 |-- file_size_bytes: long (nullable = true)
 |-- source_modified_at: timestamp (nullable = true)
 |-- content_hash: string (nullable = true)
 |-- last_seen_at: timestamp (nullable = true)
 |-- discovery_run_id: string (nullable = true)
 |-- claimed_at: timestamp (nullable = true)
 |-- lease_expires_at: timestamp (nullable = true)
 |-- last_attemp

dataset,file_name,file_path,batch_id,status,first_seen,processed_at,run_id,retry_count,error_message,data_date,file_size_bytes,source_modified_at,content_hash,last_seen_at,discovery_run_id,claimed_at,lease_expires_at,last_attempt_at,attempt_count,last_success_at,status_updated_at,source_record_count,written_record_count,rejected_record_count,processed_content_hash,schema_fingerprint,target_path,target_delta_version,processing_duration_seconds,error_type,last_error_at
customers,customers_20261014.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=14/customers_20261014.json,71a33b91-383c-4e55-aba3-a0827296a554,SUCCESS,2026-10-14T08:28:07.674Z,2026-10-14T08:35:21.498Z,128df69a-2cd4-4f62-b9ed-e4f656c0e726,0,null,2026-10-14,2042501,2026-10-14T08:25:33.000Z,853af5ff0edb2322e7fc0519657fee59b8c5f136f8c3a3516b2c14f272287c36,2026-10-14T08:27:37.044Z,DISCOVERY_20261014082737,null,null,2026-10-14T08:35:03.780Z,1,2026-10-14T08:35:21.498Z,2026-10-14T08:35:21.498Z,25000,25000,0,853af5ff0edb2322e7fc0519657fee59b8c5f136f8c3a3516b2c14f272287c36,null,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/customers/,38,null,null,null
products,products_20261014.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=14/products_20261014.json,8353ddc9-f1ac-4bf1-a0d2-4f028e36f8bf,SUCCESS,2026-10-14T08:28:07.674Z,2026-10-14T08:35:21.498Z,b90d11f8-e74f-4776-b3f2-5fe9948f8910,0,null,2026-10-14,2682482,2026-10-14T08:26:47.000Z,5c94040cd0a805cda40945fc8b48fdfc4eca8fcd42f117674753a49a8a8e1fb3,2026-10-14T08:27:37.044Z,DISCOVERY_20261014082737,2026-10-14T08:35:03.168Z,null,2026-10-14T08:35:03.168Z,1,2026-10-14T08:35:21.498Z,2026-10-14T08:35:21.498Z,25000,25000,0,5c94040cd0a805cda40945fc8b48fdfc4eca8fcd42f117674753a49a8a8e1fb3,null,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/products/,47,null,null,null
products,products_20261012.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=12/products_20261012.json,cb8f6ca4-2013-4c07-845e-b4846bce0780,SUCCESS,2026-10-12T10:16:32.827Z,2026-10-14T08:31:37.089Z,d004e230-d235-4f9e-a1a4-063d81832361,0,null,2026-10-12,445,2026-10-12T10:14:17.000Z,ff4cb6ab89299ac7957979132fcf634fd482c30414a9d48cc0e9698962e7dc06,2026-10-12T10:16:09.581Z,DISCOVERY_20261012101609,2026-10-14T08:31:27.770Z,null,2026-10-14T08:31:27.770Z,12,2026-10-14T08:31:37.089Z,2026-10-14T08:31:37.089Z,3,3,0,ff4cb6ab89299ac7957979132fcf634fd482c30414a9d48cc0e9698962e7dc06,null,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/products/,46,null,null,null
products,products_20261010.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=10/products_20261010.json,df9f6e0c-1be6-4d8b-9f6f-96ac333da35d,SUCCESS,2026-10-12T13:29:56.884Z,2026-10-14T08:31:23.394Z,d004e230-d235-4f9e-a1a4-063d81832361,0,null,2026-10-10,2779874,2026-10-12T13:26:32.000Z,2e5c605c323c2b899cee8c43c668ea58505267c122d1ecde57d4914d297c56c4,2026-10-12T13:29:31.290Z,DISCOVERY_20261012132931,2026-10-14T08:31:14.677Z,null,2026-10-14T08:31:14.677Z,9,2026-10-14T08:31:23.394Z,2026-10-14T08:31:23.394Z,25000,25000,0,2e5c605c323c2b899cee8c43c668ea58505267c122d1ecde57d4914d297c56c4,null,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/products/,45,null,null,null
products,products_20261008.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=08/products_20261008.json,d21fbcdc-4dcf-49a4-8d2c-0bfffd5b8612,SUCCESS,2026-10-11T09:38:16.334Z,2026-10-14T08:31:09.849Z,d004e230-d235-4f9e-a1a4-063d81832361,0,null,2026-10-08,2615396,2026-10-08T15:58:07.000Z,647d55b2a4e1b6a7c360248682256d5bca7c0ac36d5ce5be1d0cbaab028c0b4b,2026-10-11T09:29:27.360Z,DISCOVERY_20261011092927,2026-10-14T08

In [0]:
# ============================================================
# STEP 2 — DISCOVER LANDING FILES
# Read-only discovery; does not modify the manifest.
# ============================================================

from datetime import datetime, timezone
from pyspark.sql import functions as F

# ------------------------------------------------------------
# 1. Project configuration
# ------------------------------------------------------------

storage_account = "nestlesalesdatalake"
container = "customer-sales-datalake"

base_path = (
    f"abfss://{container}@"
    f"{storage_account}.dfs.core.windows.net/"
)

landing_base = base_path + f"landing/{source_name}/{source_name}_{dataset_name}/"
manifest_path = base_path + "metadata/file_manifest/"

discovery_run_id = (
    "DISCOVERY_"
    + datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
)

discovery_timestamp = datetime.now(timezone.utc)

print("Landing root:", landing_base)
print("Discovery run ID:", discovery_run_id)

Landing root: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/google_ads/google_ads_campaigns/
Discovery run ID: DISCOVERY_20261124070146


In [0]:
# ------------------------------------------------------------
# 2. Recursively list JSON files
# ------------------------------------------------------------

def discover_json_files(path):
    files_found = []

    for item in dbutils.fs.ls(path):
        if item.isDir():
            files_found.extend(discover_json_files(item.path))

        elif item.name.lower().endswith(".json"):
            files_found.append(item)

    return files_found

In [0]:
discovered_records = []
discovery_errors = []

dataset_path = landing_base 
print(f"\nDiscovering dataset: {dataset_name}")
try:
    items = discover_json_files(dataset_path)
    for item in items:
        try:
            # Read the file as binary to calculate a content hash.
            hash_row =(
                    spark.read
                    .format("binaryFile")
                    .load(item.path)
                    .select(
                        F.sha2(F.col("content"), 256).alias("content_hash")
                    )
                    .first()
            )                                             
            if hash_row is None or not hash_row["content_hash"]:
                raise RuntimeError("Unable to calculate SHA-256 hash.")
            modified_at = None
            if item.modificationTime:
                modified_at = datetime.fromtimestamp(
                        item.modificationTime / 1000.0,
                        tz=timezone.utc
                )                                             
            discovered_records.append({
                    "dataset": dataset_name,
                    "file_name": item.name,
                    "file_path": item.path,
                    "file_size_bytes": int(item.size),
                    "source_modified_at": modified_at,
                    "content_hash": hash_row["content_hash"],
                    "last_seen_at": discovery_timestamp,
                    "discovery_run_id": discovery_run_id
                })

        except Exception as exc:
                discovery_errors.append({
                    "dataset": dataset_name,
                    "file_path": item.path,
                    "error": f"{type(exc).__name__}: {str(exc)}"
                })

    print(f"JSON files found: {len(items)}")

except Exception as exc:
    raise RuntimeError(
            f"Could not list Landing directory: {dataset_path}"
        ) from exc 
    
print(discovered_records)


Discovering dataset: campaigns
JSON files found: 2
[{'dataset': 'campaigns', 'file_name': 'campaigns_20261029.json', 'file_path': 'abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/google_ads/google_ads_campaigns/year=2026/month=10/day=29/campaigns_20261029.json', 'file_size_bytes': 77767, 'source_modified_at': datetime.datetime(2026, 10, 29, 4, 12, 18, tzinfo=datetime.timezone.utc), 'content_hash': 'c66f4513f012c6310621b605d43d1aad76ae5518e447dc8d16024e5b98f12d25', 'last_seen_at': datetime.datetime(2026, 11, 24, 7, 1, 46, 531879, tzinfo=datetime.timezone.utc), 'discovery_run_id': 'DISCOVERY_20261124070146'}, {'dataset': 'campaigns', 'file_name': 'campaigns_20261121.json', 'file_path': 'abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/google_ads/google_ads_campaigns/year=2026/month=10/day=29/campaigns_20261121.json', 'file_size_bytes': 779972, 'source_modified_at': datetime.datetime(2026, 11, 21, 12, 8, 46, tzinfo=datetime

In [0]:
# Do not continue with a partial discovery.
if discovery_errors:
    print("Discovery errors:")
    for error in discovery_errors:
        print(error)

    raise RuntimeError(
        f"{len(discovery_errors)} file(s) could not be inspected. "
        "Resolve the errors before continuing."
    )

In [0]:
# ------------------------------------------------------------
# 3. Create the discovery DataFrame
# ------------------------------------------------------------

discovery_schema = """
    dataset STRING,
    file_name STRING,
    file_path STRING,
    file_size_bytes BIGINT,
    source_modified_at TIMESTAMP,
    content_hash STRING,
    last_seen_at TIMESTAMP,
    discovery_run_id STRING
"""

discovered_files_df = spark.createDataFrame(
    discovered_records,
    schema=discovery_schema
)                                                          
display(discovered_records) 

content_hash,dataset,discovery_run_id,file_name,file_path,file_size_bytes,last_seen_at,source_modified_at
c66f4513f012c6310621b605d43d1aad76ae5518e447dc8d16024e5b98f12d25,campaigns,DISCOVERY_20261124070146,campaigns_20261029.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/google_ads/google_ads_campaigns/year=2026/month=10/day=29/campaigns_20261029.json,77767,2026-11-24T07:01:46.531Z,2026-10-29T04:12:18.000Z
9569301c02e0333ec8654528705ccd4d872cbaab0875f0f9fa56194c95abf020,campaigns,DISCOVERY_20261124070146,campaigns_20261121.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/google_ads/google_ads_campaigns/year=2026/month=10/day=29/campaigns_20261121.json,779972,2026-11-24T07:01:46.531Z,2026-11-21T12:08:46.000Z


In [0]:
# ------------------------------------------------------------
# 4. Derive data_date
# ------------------------------------------------------------

filename_date = F.to_date(
    F.regexp_extract(
        F.col("file_name"),
        r"(\d{8})",
        1
    ),
    "yyyyMMdd"
)

year_part = F.regexp_extract(
    F.col("file_path"),
    r"/year=(\d{4})/",
    1
)                                                         
                                                          
month_part = F.regexp_extract(
    F.col("file_path"),
    r"/month=(\d{1,2})/",
    1
)

day_part = F.regexp_extract(
    F.col("file_path"),
    r"/day=(\d{1,2})/",
    1
)

partition_date = F.to_date(
    F.concat_ws(
        "-",
        year_part,
        F.lpad(month_part, 2, "0"),
        F.lpad(day_part, 2, "0")
    ),
    "yyyy-MM-dd"
)

discovered_files_df = discovered_files_df.withColumn(
    "data_date",
    F.coalesce(filename_date, partition_date)
)
display(discovered_files_df)                              

dataset,file_name,file_path,file_size_bytes,source_modified_at,content_hash,last_seen_at,discovery_run_id,data_date
campaigns,campaigns_20261029.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/google_ads/google_ads_campaigns/year=2026/month=10/day=29/campaigns_20261029.json,77767,2026-10-29T04:12:18.000Z,c66f4513f012c6310621b605d43d1aad76ae5518e447dc8d16024e5b98f12d25,2026-11-24T07:01:46.531Z,DISCOVERY_20261124070146,2026-10-29
campaigns,campaigns_20261121.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/google_ads/google_ads_campaigns/year=2026/month=10/day=29/campaigns_20261121.json,779972,2026-11-21T12:08:46.000Z,9569301c02e0333ec8654528705ccd4d872cbaab0875f0f9fa56194c95abf020,2026-11-24T07:01:46.531Z,DISCOVERY_20261124070146,2026-11-21


In [0]:
# ------------------------------------------------------------
# 5. Validate identities and dates
# ------------------------------------------------------------

duplicate_files_df = (
    discovered_files_df
    .groupBy("dataset", "file_path")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_files_df.limit(1).count() > 0:
    display(duplicate_files_df)
    raise RuntimeError("Duplicate file identities found in discovery.")

undated_files_df = discovered_files_df.filter(
    F.col("data_date").isNull()
)

if undated_files_df.limit(1).count() > 0:
    print("Files without a valid data_date:")
    display(
        undated_files_df.select(
            "dataset", "file_name", "file_path"
        )
    )

    raise RuntimeError(
        "Some files have no valid date in their filename or "
        "partition path. Fix the date extraction before registering."
    )

In [0]:
# ------------------------------------------------------------
# 6. Discovery summary
# ------------------------------------------------------------

print("\nDiscovery completed successfully.")
print("Total JSON files:", discovered_files_df.count())

display(
    discovered_files_df
    .select(
        "dataset",
        "file_name",
        "data_date",
        "file_size_bytes",
        "source_modified_at",
        "content_hash",
        "file_path"
    )
    .orderBy("dataset", "data_date", "file_name")
)



Discovery completed successfully.
Total JSON files: 2


dataset,file_name,data_date,file_size_bytes,source_modified_at,content_hash,file_path
campaigns,campaigns_20261029.json,2026-10-29,77767,2026-10-29T04:12:18.000Z,c66f4513f012c6310621b605d43d1aad76ae5518e447dc8d16024e5b98f12d25,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/google_ads/google_ads_campaigns/year=2026/month=10/day=29/campaigns_20261029.json
campaigns,campaigns_20261121.json,2026-11-21,779972,2026-11-21T12:08:46.000Z,9569301c02e0333ec8654528705ccd4d872cbaab0875f0f9fa56194c95abf020,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/google_ads/google_ads_campaigns/year=2026/month=10/day=29/campaigns_20261121.json


In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime, timezone

# 1. Validate discovery output
if "discovered_files_df" not in globals():
    raise ValueError("Run the discovery cell first.")

if discovered_files_df.count() == 0:
    raise ValueError("No files discovered. Registration stopped.")  

required_columns = {
    "dataset", "file_name", "file_path", "data_date",
    "file_size_bytes", "source_modified_at", "content_hash",
    "last_seen_at", "discovery_run_id"
}

missing = required_columns - set(discovered_files_df.columns)
if missing:
    raise ValueError(f"Discovery output is missing columns: {sorted(missing)}")

In [0]:
# 3. Confirm manifest table exists
if not DeltaTable.isDeltaTable(spark, manifest_path):
    raise ValueError(
        f"No Delta manifest found at {manifest_path}. "
        "Create the 32-column manifest table first."
    )

manifest_df = spark.read.format("delta").load(manifest_path)

if len(manifest_df.columns) != 32:
    raise ValueError(
        f"Expected 32 manifest columns, found {len(manifest_df.columns)}."
    )

In [0]:
# 5. Prepare the discovered records
now = datetime.now(timezone.utc).replace(tzinfo=None)

source = (
    discovered_files_df
    .withColumn("batch_id", F.expr("uuid()"))
    .withColumn("status", F.lit("NEW"))
    .withColumn("first_seen", F.lit(now).cast("timestamp"))
    .withColumn("processed_at", F.lit(None).cast("timestamp"))
    .withColumn("run_id", F.lit(None).cast("string"))
    .withColumn("retry_count", F.lit(0).cast("int"))
    .withColumn("error_message", F.lit(None).cast("string")) 
    .withColumn("attempt_count", F.lit(0).cast("int"))
    .withColumn("claimed_at", F.lit(None).cast("timestamp"))
    .withColumn("lease_expires_at", F.lit(None).cast("timestamp"))
    .withColumn("last_attempt_at", F.lit(None).cast("timestamp"))
    .withColumn("last_success_at", F.lit(None).cast("timestamp"))
    .withColumn("status_updated_at", F.lit(now).cast("timestamp"))
    .withColumn("source_record_count", F.lit(None).cast("long"))
    .withColumn("written_record_count", F.lit(None).cast("long"))
    .withColumn("rejected_record_count", F.lit(None).cast("long"))
    .withColumn("processed_content_hash", F.lit(None).cast("string"))
    .withColumn("schema_fingerprint", F.lit(None).cast("string"))
    .withColumn("target_path", F.lit(None).cast("string"))
    .withColumn("target_delta_version", F.lit(None).cast("long"))
    .withColumn("processing_duration_seconds", F.lit(None).cast("double"))
    .withColumn("error_type", F.lit(None).cast("string"))
    .withColumn("last_error_at", F.lit(None).cast("timestamp"))
)

manifest_columns = manifest_df.columns
source = source.select(*manifest_columns)

In [0]:
# 6. Register only previously unseen file identities
target = DeltaTable.forPath(spark, manifest_path)

(
    target.alias("t")
    .merge(
        source.alias("s"),
        "t.dataset = s.dataset AND t.file_path = s.file_path"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:

# 7. Verify registration
registered = spark.read.format("delta").load(manifest_path)

print("Registration completed.")
print("Total discovered files:", discovered_files_df.count())
print(
    "Registered NEW files:",
    registered.filter(F.col("status") == "NEW").count()
)
print("Manifest total rows:", registered.count())

display(
    registered
    .groupBy("dataset", "status")
    .count()
    .orderBy("dataset", "status")
)

display(
    registered
    .select(
        "dataset", "file_name", "data_date",
        "status", "content_hash", "batch_id"
    )
    .orderBy("dataset", "data_date", "file_name")
)


Registration completed.
Total discovered files: 2
Registered NEW files: 2
Manifest total rows: 26


dataset,status,count
campaigns,NEW,2
customers,SUCCESS,8
google_ads_campaigns,SUCCESS,2
orders,SUCCESS,7
products,SUCCESS,7


dataset,file_name,data_date,status,content_hash,batch_id
campaigns,campaigns_20261029.json,2026-10-29,NEW,c66f4513f012c6310621b605d43d1aad76ae5518e447dc8d16024e5b98f12d25,f07a6365-6f2a-4045-b5c6-856972d760f8
campaigns,campaigns_20261121.json,2026-11-21,NEW,9569301c02e0333ec8654528705ccd4d872cbaab0875f0f9fa56194c95abf020,4d88f025-7ad7-4e37-8531-896eaf16d5a7
customers,customers_20261005.json,2026-10-05,SUCCESS,64154d3f4ec59d56ac99d68032542843640de32a3e1eaea141d9f5f9404bdefd,68684d7d-4ab3-425d-b1d1-1fb84bb56b51
customers,customers_20261007.json,2026-10-07,SUCCESS,5ef2c76d178ce3e2b35c8413a647bf9f5183d6404ffca94308d02a371d9765e0,17d146be-76e6-4412-81ac-c50156373bfa
customers,customers_20261008.json,2026-10-08,SUCCESS,d82dc9b7776651362b54e203ec0734ae3a495534b7f093c9e2302e64033b5ef5,212ac11d-5416-4c17-b936-a2b6c1c37c52
customers,customers_20261009.json,2026-10-09,SUCCESS,31246e22f0a2e56887ab66100a0c69832b3c77532d3a3197a1d80f6430a3b724,b1bd8ab0-8e65-4009-a8da-2ac857abc3d7
customers,customers_20261010.json,2026-10-10,SUCCESS,8575c4da9ab0d6c00f4d34851b6152c112718589c5e2315c7f1d4ede6c6e7cc7,188fdbd0-3dc0-4c54-a847-9b2bd6ebc6f2
customers,customers_20261012.json,2026-10-12,SUCCESS,e15b6e5b19e0af1c49a8f884df23d84213be4feca76cdf4f73a08996c8f1661d,3be93bba-1424-4e49-a8b3-26570f01b932
customers,customers_20261014.json,2026-10-14,SUCCESS,853af5ff0edb2322e7fc0519657fee59b8c5f136f8c3a3516b2c14f272287c36,71a33b91-383c-4e55-aba3-a0827296a554
customers,customers_20261015.json,2026-10-15,SUCCESS,9c64b12298d5dc7a5eb1b8828e9b7c45a6a3887823e1f5162bcbfa1fc10a92c1,ab6907c4-2317-4d6f-aeee-5b4669fb1767


In [0]:

# ============================================================
# STEP 6B — CANDIDATE PREVIEW ONLY
# BRONZE NOTEBOOK OWNS CLAIMING AND PROCESSING
# ============================================================
import uuid
RUN_MODE = "INCREMENTAL"
LEASE_MINUTES = 10
# Generate a new ID for the next manifest run.
run_id = str(uuid.uuid4())

from pyspark.sql import functions as F

if "run_id" not in globals() or not run_id:
    raise ValueError("Run Step 6A first.")

manifest_df = (
    spark.read.format("delta")
    .load(manifest_path)
)

candidate_df = (
    manifest_df
    .filter(F.col("status").isin("NEW", "FAILED"))
    .select(
        "dataset",
        "file_name",
        "file_path",
        "status",
        "content_hash"
    )
    .orderBy("dataset", "file_name")
)

candidate_count = candidate_df.count()

print("Manifest discovery run ID:", run_id)
print("Eligible files awaiting Bronze processing:", candidate_count)
print("This cell does not claim or update manifest records.")

display(candidate_df)

print("STEP 6B completed.")


Manifest discovery run ID: b828da76-67e0-4e76-91a2-58b62541f923
Eligible files awaiting Bronze processing: 2
This cell does not claim or update manifest records.


dataset,file_name,file_path,status,content_hash
campaigns,campaigns_20261029.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/google_ads/google_ads_campaigns/year=2026/month=10/day=29/campaigns_20261029.json,NEW,c66f4513f012c6310621b605d43d1aad76ae5518e447dc8d16024e5b98f12d25
campaigns,campaigns_20261121.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/google_ads/google_ads_campaigns/year=2026/month=10/day=29/campaigns_20261121.json,NEW,9569301c02e0333ec8654528705ccd4d872cbaab0875f0f9fa56194c95abf020


STEP 6B completed.


In [0]:

# STEP 6D — RELEASE EXPIRED CLAIMS
# Run only after confirming no relevant jobs are active.

from delta.tables import DeltaTable
from pyspark.sql import functions as F

manifest_table = DeltaTable.forPath(spark, manifest_path)

# Capture the exact expired file identities before updating.
expired_before = (
    spark.read.format("delta").load(manifest_path)
    .filter(
        (F.col("status") == "PROCESSING")
        & F.col("lease_expires_at").isNotNull()
        & (F.col("lease_expires_at") <= F.current_timestamp())
    )
    .select("dataset", "file_path")
    .collect()
)

print("Expired claims eligible for release:", len(expired_before))

if not expired_before:
    print("No expired claims to release.")
else:
    # Reset only records that are still PROCESSING and still expired.
    (
        manifest_table.update(
            condition=(
                (F.col("status") == "PROCESSING")
                & F.col("lease_expires_at").isNotNull()
                & (F.col("lease_expires_at") <= F.current_timestamp())
            ),
            set={
                "status": F.lit("NEW"),
                "run_id": F.lit(None).cast("string"),
                "claimed_at": F.lit(None).cast("timestamp"),
                "lease_expires_at": F.lit(None).cast("timestamp"),
                "status_updated_at": F.current_timestamp()
            }
        )
    )

    print("Expired claims released. Verify the current state below.")

display(
    spark.read.format("delta").load(manifest_path)
    .filter(F.col("dataset")==dataset_name)
    .select(
        "dataset", "file_name", "status",
        "run_id", "lease_expires_at"
    )
    .orderBy("dataset", "file_name")
)


Expired claims eligible for release: 0
No expired claims to release.


dataset,file_name,status,run_id,lease_expires_at
campaigns,campaigns_20261029.json,NEW,null,null
campaigns,campaigns_20261121.json,NEW,null,null


In [0]:

from delta.tables import DeltaTable
from pyspark.sql import functions as F

# ============================================================
# STEP 7A — MANIFEST STATUS UPDATE HELPER
# ============================================================

def update_manifest_status(
    dataset,
    file_path,
    run_id,
    new_status,
    error_message=None,
    source_record_count=None,
    written_record_count=None,
    rejected_record_count=None,
    processed_content_hash=None,
    error_type=None
):
    """
    Finalize a file only when this run still owns its PROCESSING claim.
    Call SUCCESS only after verifying the Bronze write.
    """

    if new_status not in {"SUCCESS", "FAILED"}:
        raise ValueError("new_status must be SUCCESS or FAILED")

    if new_status == "SUCCESS" and not processed_content_hash:
        raise ValueError(
            "A successful file must provide its processed content hash."
        )

    target = DeltaTable.forPath(spark, manifest_path)

    update_values = {
        "status": F.lit(new_status),
        "processed_at": F.current_timestamp(),
        "status_updated_at": F.current_timestamp(),
        "error_message": F.lit(error_message).cast("string"),
        "error_type": F.lit(error_type).cast("string"),
        "source_record_count": F.lit(source_record_count).cast("long"),
        "written_record_count": F.lit(written_record_count).cast("long"),
        "rejected_record_count": F.lit(rejected_record_count).cast("long"),
        "processing_duration_seconds": F.lit(None).cast("double"),
        "last_error_at": (
            F.current_timestamp()
            if new_status == "FAILED"
            else F.col("last_error_at")
        ),
        "last_success_at": (
            F.current_timestamp()
            if new_status == "SUCCESS"
            else F.col("last_success_at")
        ),
        "processed_content_hash": (
            F.lit(processed_content_hash).cast("string")
            if new_status == "SUCCESS"
            else F.col("processed_content_hash")
        ),
        "retry_count": (
            F.coalesce(F.col("retry_count"), F.lit(0)) + 1
            if new_status == "FAILED"
            else F.col("retry_count")
        ),
        "claimed_at": F.lit(None).cast("timestamp"),
        "lease_expires_at": F.lit(None).cast("timestamp")
    }

    (
        target.alias("t")
        .update(
            condition=(
                (F.col("dataset") == F.lit(dataset))
                & (F.col("file_path") == F.lit(file_path))
                & (F.col("status") == F.lit("PROCESSING"))
                & (F.col("run_id") == F.lit(run_id))
            ),
            set=update_values
        )
    )

    # Verify the resulting state and ownership
    result = (
        spark.read.format("delta").load(manifest_path)
        .filter(
            (F.col("dataset") == dataset)
            & (F.col("file_path") == file_path)
        )
        .select("status", "run_id")
        .collect()
    )

    if (
        len(result) != 1
        or result[0]["status"] != new_status
        or result[0]["run_id"] != run_id
    ):
        raise RuntimeError(
            f"Could not verify {new_status} for {file_path}. "
            "The row may no longer be owned by this run."
        )

    print(f"Manifest updated: {dataset} | {new_status} | {file_path}")
